Compare read depth and gene count from major human NSC datasets (Dumitru, Franjic, Wang, Liu)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc

# Dumitru et al.

In [ ]:
dumitru_adata = sc.read_h5ad('../data/dumitru/single_nucleus/h5ad/dumitru_all_donors_nsc_annotations.h5ad')
dumitru_adata

In [ ]:
sc.pp.calculate_qc_metrics(dumitru_adata, inplace=True)

In [ ]:
dumitru_adata.obs['total_counts'].describe()

In [ ]:
dumitru_adata.obs['total_counts'].median()

In [ ]:
dumitru_adata.obs['n_genes_by_counts'].describe()

In [ ]:
dumitru_adata.obs['n_genes_by_counts'].median()

# Franjic et al.

In [ ]:
# Read matrix market format
franjic_adata = sc.read_mtx('../data/franjic/GSE186538_Human_counts.mtx').T 

# Read gene names
genes = pd.read_csv('../data/franjic/GSE186538_Human_genes.txt', header=None, sep='\t')
franjic_adata.var_names = genes[0].values

# Read cell metadata
obs = pd.read_csv('../data/franjic/GSE186538_Human_cell_meta.txt', sep='\t', index_col=0)
franjic_adata.obs = obs
franjic_adata

In [ ]:
sc.pp.calculate_qc_metrics(franjic_adata, inplace=True)

In [ ]:
franjic_adata.obs['total_counts'].describe()

In [ ]:
franjic_adata.obs['total_counts'].median()

In [ ]:
franjic_adata.obs['n_genes_by_counts'].describe()

In [ ]:
franjic_adata.obs['n_genes_by_counts'].median()

# Wang et al.

In [ ]:
wang_adata = sc.read_csv('../data/wang/GSE163737_HHP_gene_expression_matrix.txt', delimiter='\t', first_column_names=True).T
wang_adata

In [ ]:
sc.pp.calculate_qc_metrics(wang_adata, inplace=True)

In [ ]:
wang_adata

In [ ]:
wang_adata.obs['total_counts'].describe()

In [ ]:
wang_adata.obs['total_counts'].median()

In [ ]:
wang_adata.obs['n_genes_by_counts'].describe()

In [ ]:
wang_adata.obs['n_genes_by_counts'].median()

# Liu et al.

In [ ]:
liu_adata = sc.read_h5ad('../data/liu/hNSPC_raw_counts.h5ad')
liu_adata

In [ ]:
sc.pp.calculate_qc_metrics(liu_adata, inplace=True)

In [ ]:
liu_adata.obs['total_counts'].describe()

In [ ]:
liu_adata.obs['total_counts'].median()

In [ ]:
liu_adata.obs['n_genes_by_counts'].describe()

In [ ]:
liu_adata.obs['n_genes_by_counts'].median()

# Violin plots

In [ ]:
# Combine all datasets into one AnnData object for plotting
combined_adata_list = []

for name, adata in [('Dumitru', dumitru_adata), 
                     ('Franjic', franjic_adata), 
                     ('Wang', wang_adata), 
                     ('Liu', liu_adata)]:
    adata.obs['dataset'] = name
    combined_adata_list.append(adata)

combined_adata = sc.concat(combined_adata_list)

# enforce plotting order
dataset_order = ['Franjic', 'Dumitru', 'Wang', 'Liu']
combined_adata.obs['dataset'] = pd.Categorical(
    combined_adata.obs['dataset'],
    categories=dataset_order,
    ordered=True)

In [ ]:
# Read depth violin plot
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 5))
sc.pl.violin(combined_adata, keys='total_counts', groupby='dataset', inner='box', stripplot=False, ax=ax, show=False)
ax.set_ylim(0, 2000000)
ax.ticklabel_format(style='plain', axis='y')
ax.set_title('Read Depth Across Datasets')
plt.show()

In [ ]:
# Gene counts violin plot
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 5))
sc.pl.violin(combined_adata, keys='n_genes_by_counts', groupby='dataset', inner='box', stripplot=False, ax=ax, show=False)
ax.set_title('Gene Counts Across Datasets')
plt.show()